# Tekne Dedektörü v4s_frozen_v3 — Google Colab GPU Eğitimi (2. genişletme: newVideoShip)

Bu notebook, aynı tarifle (YOLOv8s, backbone donuk `freeze=10`, imgsz=640, patience=25) ama
şimdi **newVideoShip'teki 17 videonun elle etiketlenmesiyle ikinci kez genişletilmiş datasetle**
(train: 3944 -> 8603, val: 1241 -> 2097, toplam 5185 -> 10700 görüntü) baştan (fresh, resume değil) eğitir.

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir > T4 GPU** (veya A100/L4) seç.
2. `boat_v4s_frozen_v3_bundle.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, ~671MB) Google Drive'ına yükle.
3. Aşağıdaki hücreleri sırayla çalıştır.

In [ ]:
# 1) GPU kontrolü
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip dosyasını Drive'dan al ve aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/boat_v4s_frozen_v3_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml içindeki path'i Colab'daki yeni konuma göre düzelt
# (Mac'te: /Users/armin/Desktop/depth-anything/yolo_dataset_v4 idi)
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) backbone'un gercekten ilk 10 katmanda bittigini teyit et (Mac'te de dogrulanmisti)
from ultralytics import YOLO

_check = YOLO('/content/work/yolov8s.pt')
for i, layer in enumerate(_check.model.model):
    print(i, layer.__class__.__name__)

In [ ]:
# 7) Eğitim — boat_v4s_frozen ile AYNI tarif (freeze=10, imgsz=640, patience=25),
# tek fark: 2x genisletilmis dataset (v1 hard-mining + newVideoShip elle etiketleme). Fresh baslatiyoruz (resume degil) ki eski/yeni
# dataset karsilastirmasi ayni recipe uzerinden temiz olsun.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=640,
    device=0,
    batch=16,
    patience=25,
    freeze=10,
    project='/content/work/runs_boat_yolo',
    name='boat_v4s_frozen_v3',
    verbose=True,
)

In [ ]:
# 8) Eğitim koptuysa devam ettirmek için (7. hücre yerine bunu çalıştır):
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v4s_frozen_v3/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 9) Bitince: sonuçları Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_v3_results
!cp -r /content/work/runs_boat_yolo/boat_v4s_frozen_v3 /content/drive/MyDrive/boat_v4s_frozen_v3_results/
print('Kopyalandı: Google Drive > boat_v4s_frozen_v3_results > boat_v4s_frozen_v3')

## Eğitim bitince Mac'ine geri alma

1. Drive'daki `boat_v4s_frozen_v3_results/boat_v4s_frozen_v3` klasörünü indir (`weights/best.pt` şart, geri kalanı - grafikler/results.csv - isteğe bağlı ama karşılaştırma için faydalı).
2. Bana zip'i ilet, ben `runs/detect/runs_boat_yolo/boat_v4s_frozen_v3/` altına yerleştirip önceki denemelerle (dar veri / v2 genişletme) karşılaştırırım.

**Not — Colab'ın ücretsiz kotası:** Oturumlar genelde ~12 saatte bir kesilir, uzun süre etkileşimsiz kalırsan daha erken de kopabilir. Dataset artık 2 katından fazla büyüdüğü (10700 görüntü) için epoch süresi öncekinden (74 epoch ~50 dk) daha uzun sürebilir.